# TP 1 — Exercice 5 : Histogramme d'une image (Niveaux de gris et Couleurs)

1. Calculer les histogrammes avec `cv2.calcHist()` pour `tp1_tp2_cameraman_gray.bmp` (`im1`) et `tp1_exo4_exo5_peppers_color.bmp` (`im2`).
2. Afficher les histogrammes avec Matplotlib.
3. Implémenter l'algorithme personnel `calcul_histogramme_perso()` et vérifier l'égalité exacte avec OpenCV.

In [ ]:
from pathlib import Path
import math
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt

def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, cwd.parent, cwd.parent.parent):
        if (candidate / "tp1_tp2_images").is_dir():
            return candidate
    raise FileNotFoundError(
        "Impossible de trouver le dossier 'tp1_tp2_images'. "
        "Lancez Jupyter depuis la racine TAI ou depuis tp1_labs / tp2_labs."
    )

ROOT_DIR = find_project_root()
IMG_DIR = ROOT_DIR / "tp1_tp2_images"
OUT_DIR = ROOT_DIR / "tp1_labs" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def load_image(filename, flags=cv2.IMREAD_COLOR):
    path = IMG_DIR / filename
    img = cv2.imread(str(path), flags)
    if img is None:
        raise FileNotFoundError(f"Image introuvable ou illisible: {path}")
    return img

def show_images(items, cols=3, figsize_per_cell=(5, 4)):
    n = len(items)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * figsize_per_cell[0], rows * figsize_per_cell[1]))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes[n:]:
        ax.axis("off")
    for ax, (title, img) in zip(axes, items):
        if img.ndim == 2:
            ax.imshow(img, cmap="gray", vmin=0, vmax=255)
        elif img.shape[2] == 3:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        elif img.shape[2] == 4:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGRA2RGBA))
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

print(f"ROOT_DIR : {ROOT_DIR}")
print(f"IMG_DIR  : {IMG_DIR} ({len(list(IMG_DIR.iterdir()))} images)")
print(f"OUT_DIR  : {OUT_DIR}")

In [ ]:
# 1) Lecture des images
im1 = load_image("tp1_tp2_cameraman_gray.bmp", cv2.IMREAD_GRAYSCALE)
im2 = load_image("tp1_exo4_exo5_peppers_color.bmp", cv2.IMREAD_COLOR)

# 2) Calcul avec cv2.calcHist()
hist1 = cv2.calcHist([im1], [0], None, [256], [0, 256]).ravel()
hist2_B = cv2.calcHist([im2], [0], None, [256], [0, 256]).ravel()
hist2_G = cv2.calcHist([im2], [1], None, [256], [0, 256]).ravel()
hist2_R = cv2.calcHist([im2], [2], None, [256], [0, 256]).ravel()

# 4) Algorithme personnel
def calcul_histogramme_perso(canal_2d):
    hist = np.zeros(256, dtype=np.float32)
    h, w = canal_2d.shape
    for y in range(h):
        for x in range(w):
            hist[canal_2d[y, x]] += 1.0
    return hist

histp1 = calcul_histogramme_perso(im1)
histp2_B = calcul_histogramme_perso(im2[:, :, 0])
histp2_G = calcul_histogramme_perso(im2[:, :, 1])
histp2_R = calcul_histogramme_perso(im2[:, :, 2])

# Vérification stricte
print("Correspondance Cameraman NG :", np.array_equal(hist1, histp1))
print("Correspondance Peppers B    :", np.array_equal(hist2_B, histp2_B))
print("Correspondance Peppers G    :", np.array_equal(hist2_G, histp2_G))
print("Correspondance Peppers R    :", np.array_equal(hist2_R, histp2_R))

# 3) Affichage Matplotlib
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes[0, 0].imshow(im1, cmap="gray", vmin=0, vmax=255)
axes[0, 0].set_title("im1 : Cameraman (Niveaux de gris)")
axes[0, 0].axis("off")

axes[0, 1].plot(hist1, color="black", label="OpenCV calcHist")
axes[0, 1].plot(histp1, "--", color="orange", label="Perso")
axes[0, 1].set_title("Histogramme im1 (NG)")
axes[0, 1].set_xlim([0, 255])
axes[0, 1].set_xlabel("Niveau de gris")
axes[0, 1].set_ylabel("Nombre de pixels")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

axes[1, 0].imshow(cv2.cvtColor(im2, cv2.COLOR_BGR2RGB))
axes[1, 0].set_title("im2 : Peppers (Couleur)")
axes[1, 0].axis("off")

axes[1, 1].plot(hist2_R, color="red", label="Rouge (R)")
axes[1, 1].plot(hist2_G, color="green", label="Vert (G)")
axes[1, 1].plot(hist2_B, color="blue", label="Bleu (B)")
axes[1, 1].set_title("Histogrammes RGB de im2")
axes[1, 1].set_xlim([0, 255])
axes[1, 1].set_xlabel("Intensité")
axes[1, 1].set_ylabel("Nombre de pixels")
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()